# Image → embeddings → PCA latents

Extract image embeddings from a chosen pretrained encoder (AlexNet fc6, CLIP, ...) for every image in a dataset, reduce to the top `N_COMPONENTS` principal components, and save the result as a fixed per-image latent — a drop-in replacement for the β-VAE latent used to condition the diffusion U-Net (`forward(x, t, z)`).

Pick the encoder in **Config** below via `MODEL_NAME` — everything downstream (preprocessing, output directory, saved artifacts) adapts automatically. Add new encoders by adding an entry to `MODEL_REGISTRY` in the "Load encoder model" section.

PCA is fit on a random subsample (fits in RAM), then applied to the full dataset in a single streaming pass that never materializes all activations at once — see "Stream the full train/val sets" below. Latents are written into `OUTPUT_DIR` (named after the chosen encoder) as `train_latents.npy` / `val_latents.npy`, row-aligned with the source image directories, for downstream notebooks/scripts to load directly (e.g. `np.load(...)`).

In [1]:
import os
import sys
import json

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import models, transforms
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import joblib
import matplotlib.pyplot as plt

In [2]:
REPO_ROOT = ".."
sys.path.insert(0, REPO_ROOT)

import importlib.metadata
if not hasattr(importlib.metadata, '_patched'):

    _original_version = importlib.metadata.version
    def _patched_version(name):
        try:
            return _original_version(name)
        except importlib.metadata.PackageNotFoundError:
            return 'unknown'
    importlib.metadata.version = _patched_version
    importlib.metadata._patched = True

# from data_utils import build_tiny_imagenet_dataloaders
from data_utils import build_encode_dataloader

## Config

In [3]:
# holds train_images.npy / val_images.npy
# DATA_DIR = "/content/data/imagenet64"
# DATA_DIR = os.path.join(REPO_ROOT, "data/tiny-imagenet-200")
# DATA_DIR = os.path.join(REPO_ROOT, "data/500Stimuli")
TRAIN_DATA_DIR = os.path.join(REPO_ROOT, "data/15901Stimuli")
VAL_DATA_DIR = os.path.join(REPO_ROOT, "data/500Stimuli")

# which encoder to extract embeddings from — see MODEL_REGISTRY in "Load encoder model" below
# options: "alexnet_fc6", "clip_vit_b32", "clip_vit_l14"
MODEL_NAME = "clip_vit_b32"

# default input resolution; overridden per-model once MODEL_REGISTRY is defined below
IMAGE_SIZE = 224
BATCH_SIZE = 64

# latent_dim to use in the diffusion config
N_COMPONENTS = 50

# random train images used to fit PCA (fits comfortably in RAM)
SUBSAMPLE_SIZE = 15000
SEED = 42

# OUTPUT_DIR = os.path.join(REPO_ROOT, "data/imagenet64_alexnet_fc6_pca_latents")
OUTPUT_DIR = os.path.join(REPO_ROOT, f"data/stimuli_{MODEL_NAME}_pca_latents")

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# subprocess.run([
#     "python", os.path.join(REPO_ROOT, "scripts/prepare_imagenet64_dataset.py"),
#     "--drive_train_part1_zip", os.path.join(REPO_ROOT, "data/imagenet64/Imagenet64_train_part1.zip"),
#     "--drive_train_part2_zip", os.path.join(REPO_ROOT, "data/imagenet64/Imagenet64_train_part2.zip"),
#     "--drive_val_zip", os.path.join(REPO_ROOT, "data/imagenet64/Imagenet64_val.zip"),
#     "--data_dir", DATA_DIR,
# ], check=True)

## Load encoder model

Each encoder is registered in `MODEL_REGISTRY` below as `image_size` (native input resolution) plus a `load_fn(device)` that returns `(extract_batch_fn, cleanup_fn)`:
- `extract_batch_fn(batch) -> np.ndarray` runs one batch through the model and returns its raw (pre-PCA) embeddings.
- `cleanup_fn()` releases any resources (e.g. forward hooks) once extraction is done.

To add a new encoder, write a loader function following this same signature and add an entry to `MODEL_REGISTRY`.

### AlexNet fc6 (relu6)

`classifier` is `[Dropout, Linear(9216→4096) fc6, ReLU, Dropout, Linear(4096→4096) fc7, ReLU, Linear(4096→1000)]`.
"fc6 activations" conventionally means the *post-ReLU* output (`relu6`), i.e. the output of `classifier[2]` — that's what the hook below captures.

In [4]:
def _load_alexnet_fc6(device):
    """AlexNet fc6 (relu6) activations, 4096-d. See markdown above for which layer this is."""
    alexnet = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1).to(device).eval()
    for p in alexnet.parameters():
        p.requires_grad_(False)

    _activation = {"value": None}

    def _fc6_hook(module, inp, out):
        _activation["value"] = out.detach()

    hook_handle = alexnet.classifier[2].register_forward_hook(_fc6_hook)  # relu6 output

    # AlexNet's pretrained weights expect ImageNet-normalized input; build_encode_dataloader
    # only resizes and gives [0,1] tensors, so normalize per-batch here.
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    @torch.no_grad()
    def extract_batch(batch) -> np.ndarray:
        """Run one batch through AlexNet, return its fc6 (relu6) activations as a numpy array.
        O(1) memory regardless of dataset size — only ever holds one batch's activations.

        The isinstance guard is defensive: build_encode_dataloader always yields plain image
        tensors (never (image, latent) tuples), unlike the ImageNet64 dataloader used in the
        Colab counterpart of this notebook.
        """
        images = batch[0] if isinstance(batch, (list, tuple)) else batch
        images = normalize(images).to(device)
        alexnet(images)
        return _activation["value"].cpu().numpy()

    def cleanup():
        hook_handle.remove()

    return extract_batch, cleanup

### CLIP image embeddings

Uses the HuggingFace `transformers` CLIP implementation — no extra dependency beyond what's already installed. Returns the post-projection image embedding (512-d for ViT-B/32, 768-d for ViT-L/14). Each CLIP checkpoint ships its own normalization stats (`CLIPImageProcessor.image_mean`/`image_std`), fetched here rather than hardcoded so different CLIP variants stay correct.

In [5]:
def _make_clip_loader(model_id: str):
    """Return a load_fn(device) for a given CLIP checkpoint (e.g. "openai/clip-vit-base-patch32")."""
    def load_fn(device):
        from transformers import CLIPModel, CLIPImageProcessor

        clip_model = CLIPModel.from_pretrained(model_id).to(device).eval()
        for p in clip_model.parameters():
            p.requires_grad_(False)

        processor = CLIPImageProcessor.from_pretrained(model_id)
        normalize = transforms.Normalize(mean=processor.image_mean, std=processor.image_std)

        @torch.no_grad()
        def extract_batch(batch) -> np.ndarray:
            images = batch[0] if isinstance(batch, (list, tuple)) else batch
            images = normalize(images).to(device)
            features = clip_model.get_image_features(pixel_values=images)
            if not isinstance(features, torch.Tensor):
                # Look for 'image_embeds' (CLIPModel) or 'pooler_output' (CLIPVisionModel)
                features = getattr(features, "image_embeds", getattr(features, "pooler_output", features[0]))

            return features.cpu().numpy()

        def cleanup():
            pass

        return extract_batch, cleanup

    return load_fn

In [9]:
MODEL_REGISTRY = {
    "alexnet_fc6": {
        "display_name": "AlexNet fc6 (relu6)",
        "image_size": 224,
        "load_fn": _load_alexnet_fc6,
    },
    "clip_vit_b32": {
        "display_name": "CLIP ViT-B/32 (openai/clip-vit-base-patch32)",
        "image_size": 224,
        "load_fn": _make_clip_loader("openai/clip-vit-base-patch32"),
    },
    "clip_vit_l14": {
        "display_name": "CLIP ViT-L/14 (openai/clip-vit-large-patch14)",
        "image_size": 224,
        "load_fn": _make_clip_loader("openai/clip-vit-large-patch14"),
    },
}

assert MODEL_NAME in MODEL_REGISTRY, f"Unknown MODEL_NAME {MODEL_NAME!r}, choose from {list(MODEL_REGISTRY)}"
IMAGE_SIZE = MODEL_REGISTRY[MODEL_NAME]["image_size"]

extract_batch, _cleanup_model = MODEL_REGISTRY[MODEL_NAME]["load_fn"](device)
print(f"Loaded encoder: {MODEL_REGISTRY[MODEL_NAME]['display_name']}  (image_size={IMAGE_SIZE})")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

Loaded encoder: CLIP ViT-B/32 (openai/clip-vit-base-patch32)  (image_size=224)


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

## Build train/val dataloaders (unshuffled, for encoding)

`build_encode_dataloader` returns a single, always-unshuffled `DataLoader` over a flat image directory, so index `i` here matches row `i` of the row order the streaming pass below writes latents back into. Called once each for `TRAIN_DATA_DIR` and `VAL_DATA_DIR`.

In [10]:
# train_encode_dl, val_encode_dl = build_imagenet64_dataloaders(
#     DATA_DIR, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle_train=False, num_workers=0,
# )

# train_encode_dl, val_encode_dl = build_tiny_imagenet_dataloaders(
#     DATA_DIR, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle_train=False, num_workers=0,
# )

train_encode_dl = build_encode_dataloader(
    TRAIN_DATA_DIR, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, num_workers=0
)

val_encode_dl = build_encode_dataloader(
    VAL_DATA_DIR, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, num_workers=0
)

Encode set: 15901 images (249 batches)
Encode set: 500 images (8 batches)


## Fit PCA on a random subsample of train

PCA's top-`N_COMPONENTS` subspace is stable with far fewer than all 1.28M images. Fit it on a random subsample that fits comfortably in RAM, then apply the fitted transform to the full dataset in the streaming pass further down — the full `[N, D]` embedding matrix is never materialized.

In [11]:
n_train = len(train_encode_dl.dataset)
rng = np.random.default_rng(SEED)
subsample_idx = np.sort(rng.choice(n_train, size=min(SUBSAMPLE_SIZE, n_train), replace=False)).tolist()

subsample_dl = DataLoader(
    Subset(train_encode_dl.dataset, subsample_idx),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True,
)

subsample_feats = np.concatenate([extract_batch(batch) for batch in subsample_dl], axis=0)
print(f"Subsample {MODEL_NAME} embeddings: {subsample_feats.shape}")

/home/jhanliufu/RutishauserLab/DiffusionBANMap/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

## Standardize + PCA

Raw embeddings can have very different per-dimension scales (e.g. AlexNet fc6 is post-ReLU and non-negative with uneven unit-wise variance), so z-score before PCA — otherwise a handful of high-variance dimensions would dominate the top PCs. Fit on the random subsample extracted above.

In [ ]:
scaler = StandardScaler()
feats_scaled = scaler.fit_transform(subsample_feats)

pca = PCA(n_components=N_COMPONENTS, random_state=SEED, svd_solver="randomized")

In [ ]:
pca.fit(feats_scaled)

print(f"Fitted PCA on {subsample_feats.shape[0]:,} images")
print(f"Explained variance (top {N_COMPONENTS} PCs): {pca.explained_variance_ratio_.sum():.3f}")

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel("# PCs")
plt.ylabel("cumulative explained variance")
plt.title(f"{MODEL_NAME} → PCA")
plt.tight_layout()
plt.show()

## Stream the full train/val sets through the encoder → PCA

One batch of raw embeddings lives in memory at a time — each batch is transformed (`scaler` → `pca`) immediately and its `N_COMPONENTS`-d rows written into a preallocated latents array on disk, so the full `[N, D]` embedding matrix is never materialized. Written into `OUTPUT_DIR` as `train_latents.npy` / `val_latents.npy`, row-aligned with the corresponding source image directory.

This is the expensive, unavoidable part (one forward pass per image through the encoder), run this on a GPU runtime if at all possible. Resumable: re-running this cell skips any split whose latents file already exists (atomic tmp-then-rename, same pattern as `prepare_imagenet64_dataset.py`).

In [ ]:
def _stream_latents(encode_dl: DataLoader, out_path: str) -> None:
    if os.path.exists(out_path):
        print(f"Already extracted: {out_path}")
        return
    n = len(encode_dl.dataset)
    tmp_path = out_path + ".tmp"
    out = np.lib.format.open_memmap(tmp_path, mode="w+", dtype=np.float32, shape=(n, N_COMPONENTS))
    offset = 0
    for batch in encode_dl:
        feat_batch = extract_batch(batch)
        z = pca.transform(scaler.transform(feat_batch)).astype(np.float32)
        out[offset:offset + z.shape[0]] = z
        offset += z.shape[0]
        if offset % (BATCH_SIZE * 100) < BATCH_SIZE or offset == n:
            print(f"  {offset:,}/{n:,} latents written → {out_path}")
    out.flush()
    del out
    os.rename(tmp_path, out_path)
    print(f"Done: {out_path}")

In [ ]:
train_latents_path = os.path.join(OUTPUT_DIR, "train_latents.npy")
val_latents_path = os.path.join(OUTPUT_DIR, "val_latents.npy")

_stream_latents(train_encode_dl, train_latents_path)
_stream_latents(val_encode_dl, val_latents_path)

_cleanup_model()

## Save PCA/scaler artifacts + manifest

Latents themselves already live in `OUTPUT_DIR` (used positionally by the trainer, see below). This saves the fitted `StandardScaler`/`PCA` — for mapping any new image into the same latent space later — and a manifest, to a repo-relative, more persistent location.

**Note:** if `OUTPUT_DIR` lives on Colab's local runtime disk (`/content/...`), copy `train_latents.npy` / `val_latents.npy` to Drive if you want them to survive a runtime restart; regenerating them means re-running the streaming pass above.

In [ ]:
joblib.dump(scaler, os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_scaler.joblib"))
joblib.dump(pca, os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_pca.joblib"))

with open(os.path.join(OUTPUT_DIR, "manifest.json"), "w") as f:
    json.dump({
        "model_name": MODEL_NAME,
        "image_size": IMAGE_SIZE,
        "train_data_dir": TRAIN_DATA_DIR,
        "val_data_dir": VAL_DATA_DIR,
        "n_train": len(train_encode_dl.dataset),
        "n_val": len(val_encode_dl.dataset),
        "subsample_size": int(subsample_feats.shape[0]),
        "n_components": N_COMPONENTS,
        "explained_variance_ratio_sum": float(pca.explained_variance_ratio_.sum()),
        "train_latents_path": train_latents_path,
        "val_latents_path": val_latents_path,
    }, f, indent=2)

print(f"Saved scaler + PCA + manifest → {OUTPUT_DIR}")
print(f"Saved latents → {train_latents_path}, {val_latents_path}")

## Using these latents

For the stimuli sets used here, load `OUTPUT_DIR/val_latents.npy` / `train_latents.npy` directly with `np.load(...)` in downstream analysis notebooks (e.g. `analysis/vae_neuron_alignment.ipynb`).

To condition diffusion training on these latents instead, point `TRAIN_DATA_DIR`/`VAL_DATA_DIR` at an ImageNet64-format directory (containing `train_images.npy`/`val_images.npy`) and set `OUTPUT_DIR` to that same directory, so `build_imagenet64_dataloaders` (`data_utils.py`) auto-detects `train_latents.npy`/`val_latents.npy` next to them and yields `(image, latent)` pairs instead of plain image tensors — see the Colab counterpart of this notebook (`extract_image_embeddings_colab.ipynb`), which is set up for exactly that pipeline. Set `latent_dim: {N_COMPONENTS}` in the diffusion config to match.

`DiffusionTrainer` (`scripts/diffusion_trainer.py`) auto-detects the same way — `_unpack_batch` checks whether a batch is a tuple, and if so feeds that latent straight into `_get_conditioning` instead of the unconditional-zeros / on-the-fly-VAE-encode path. Set `cfg_uncond_prob > 0` in the config if you also want classifier-free guidance usable at sampling time.